In [ ]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


#### I compared the station_name but didn't compare the native_id. While the raws insertaion based on native_id to recongnize station. 

Because I didn't compare native_id before inserting, and the native_id exist many mistach, thus caused some confusion.

It can be mainly devided into two parts, the first batch is different id, same station_name; the second batch is different id, different 


Thus, only the data with same native_id has been inserted and merged; data with different native_id are recognized as different stations.

### Match stations, using station name, lat, lon. The name matching in Raw data meta_data form, in the folder and in the database

In [ ]:
from crmprtd import Row
import logging
import os
import pickle
import pandas as pd
import numpy as np
from natsort import natsorted
from natsort import natsort_keygen
from pprint import pprint
from collections import Counter
from collections import defaultdict

import re
from rapidfuzz import fuzz
from crmprtd import infer
from fern_func import *
# from sql_func import *
import sqlalchemy as sa
from fern_raw_db_dompare import *

In [ ]:
engine = sa.create_engine("postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass", echo=False)
session = Session(engine)

In [ ]:
# --- your SQL query ---
query = sa.text("""
    SELECT DISTINCT 
        m.station_name, 
        m.lat,
        m.lon,
        m.elev,
        s.native_id, 
        s.mod_user
    FROM meta_history AS m
    JOIN meta_station AS s 
        ON m.station_id = s.station_id
    WHERE s.network_id = 11;
""")

# --- execute and read into a pandas DataFrame ---
with engine.connect() as conn:
    df = pd.read_sql(query, conn)
# df now contains the result

df_multi = df[df.groupby("station_name")["native_id"].transform("nunique") > 1]
df_multi = df_multi.reset_index(drop=True)

order = {"crmp": 0, "tongli1997": 1}

df_multi = (
    df_multi
    .assign(sort_key=df_multi["mod_user"].map(order).fillna(2))
    .sort_values(by=["station_name", "sort_key"])
    .drop(columns="sort_key")
    .reset_index(drop=True)
)
df_multi

,station_name,lat,lon,elev,native_id,mod_user
0,CLAYTON FALLS,52.279579,-126.890543,1260.0,3C10P,tongli1997
1,CLAYTON FALLS,52.279900,-126.890200,1275.0,1199,tongli1997
2,EP1104.02 Quesnel Highland (ESSFwc3),52.609233,-121.411100,1558.0,Blackbear,crmp
3,EP1104.02 Quesnel Highland (ESSFwc3),52.686367,-121.201533,1518.0,Upper_Grain_Creek,crmp
4,EP1104.02 Quesnel Highland (ESSFwc3),52.676367,-121.186733,1621.0,Lower_Grain_Creek,crmp


In [ ]:
# --- Main workflow ---
# --- read data ---
meta_path = '/workspaces/crmprtd/fern/FERNNorth2025_insert/tables/20241125 MeteorologicalNetworks-FERN-VF-shared.xlsx'

df_station = pd.read_excel(meta_path)
df_station = df_station.sort_values(by='station_name', key=natsort_keygen())


# --- output folder ---
output_folder = '/workspaces/crmprtd/fern/FERNNorth2025_insert/output/'
os.makedirs(output_folder, exist_ok=True)


In [ ]:

native_ids = df_station['native_id'].tolist()
station_names = df_station['station_name'].tolist()

query = sa.text("""
    SELECT DISTINCT m.station_name, s.native_id
    FROM meta_history AS m
    JOIN meta_station AS s ON m.station_id = s.station_id
    WHERE s.network_id = 11
      AND m.station_name = ANY(:station_names)
""")

with engine.connect() as conn:
    existing_stations = pd.read_sql(query, conn, params={"station_names": station_names})

print("Stations found in DB:")
existing_stations

Stations found in DB:


,station_name,native_id
0,Atlin School,AtlSch
1,BarrenWx,Barren
2,BlackhawkWx,Blackhawk
3,BoulderWx,Boulder Creek
4,BowronPit,BowronPit
5,BulkleyWx,PGTIS1
6,Canoe Mountain Stn,Canoe Mountain Stn
7,Caribou Pass,CaribouPass
8,ChapmanWx,Chapman
9,ChiefLakeWx,ChiefWx


In [ ]:
existing_native_ids = existing_stations['native_id'].tolist()

query = sa.text("""
    SELECT 
        m.station_name,
        s.native_id,
        m.lat,
        m.lon,
        m.elev,
        v.net_var_name, 
        v.unit,
        MIN(o.obs_time) AS earliest_time,
        MAX(o.obs_time) AS latest_time
    FROM obs_raw AS o
    JOIN meta_history AS m ON o.history_id = m.history_id
    JOIN meta_vars AS v ON o.vars_id = v.vars_id
    JOIN meta_station AS s ON m.station_id = s.station_id
    WHERE s.network_id = 11
      AND s.native_id = ANY(:native_ids)
    GROUP BY v.net_var_name, v.unit, m.station_name, m.lat, m.lon, s.native_id, m.elev
    ORDER BY v.net_var_name;
""")

with engine.connect() as conn:
    db_summary = pd.read_sql(query, conn, params={"native_ids": existing_native_ids})

db_summary

,station_name,native_id,lat,lon,elev,net_var_name,unit,earliest_time,latest_time
0,BarrenWx,Barren,54.510444,-126.614341,1051.0,DewPt_30cm,celsius,2021-07-09 12:00:00,2025-09-24 12:00:00
1,BoulderWx,Boulder Creek,55.108173,-127.374740,385.0,DewPt_30cm,celsius,2020-07-07 17:00:00,2025-09-12 17:00:00
2,BowronPit,BowronPit,53.902111,-122.015389,722.0,DewPt_30cm,celsius,2018-08-29 15:00:00,2025-06-11 12:00:00
3,ChapmanWx,Chapman,54.883903,-126.623533,848.0,DewPt_30cm,celsius,2021-07-08 12:00:00,2025-09-24 08:00:00
4,CoalmineWx,CoalmineWx,53.741139,-121.784639,840.0,DewPt_30cm,celsius,2018-08-29 14:00:00,2025-09-05 09:00:00
...,...,...,...,...,...,...,...,...,...
470,SeebachWx,Seebach,54.355000,122.058000,780.0,WindSpeedms,m s-1,2023-07-13 12:00:00,2025-09-09 10:00:00
471,Sunbeam,Mcbride Mountain Stn,53.338690,-120.121080,2000.0,WindSpeedms,m s-1,2009-09-09 16:00:00,2026-01-06 09:00:00
472,Tamarac,Bednesti,53.871334,-123.370441,865.0,WindSpeedms,m s-1,2021-07-29 13:00:00,2025-10-02 15:00:00
473,ThompsonWx,Thompson,54.333115,-126.099445,869.0,WindSpeedms,m s-1,2007-09-12 16:00:00,2025-09-22 10:00:00


In [ ]:
raw_summary_path = os.path.join(output_folder, "Fern_raw_station_variable_summary.csv")

raw_summary = pd.read_csv(raw_summary_path)

all_matches = []

for station_name in df_station["station_name"]:
    raw_one = raw_summary.loc[raw_summary["station_name"] == station_name]
    db_one = db_summary.loc[db_summary["station_name"] == station_name]

    station_matches = fuzzy_match_station_vars(raw_one, db_one, threshold=80)
    all_matches.append(station_matches)

# Combine all station matches
# Filter out empty DataFrames before concatenation
non_empty_matches = [df for df in all_matches if not df.empty]

if non_empty_matches:
    matches_df = pd.concat(non_empty_matches, ignore_index=True)
else:
    matches_df = pd.DataFrame(columns=["station_name", "variable", "db_var_match", "score"])
    
# Merge with raw_summary
raw_summary_matched = raw_summary.merge(matches_df, on=["station_name", "variable"], how="left")

In [ ]:
#### merging the information in the database into the same match form
raw_summary_enriched = raw_summary_matched.merge(
    db_summary[['station_name','native_id', 'net_var_name', 'unit', 'earliest_time', 'latest_time']],
    left_on=['station_name', 'db_var_match'],   # match station + variable
    right_on=['station_name', 'net_var_name'],
    how='left'
)
raw_summary_enriched = raw_summary_enriched.drop(columns=['net_var_name'])

raw_summary_enriched = raw_summary_enriched.rename(
    columns={
        "unit_x": "unit_raw",
        'native_id_x':'native_id_raw',
        'native_id_y': 'native_id_db',
        "earliest_time_x": "earliest_time_raw",
        "latest_time_x": "latest_time_raw",
        "unit_y": "unit_db",
        "earliest_time_y": "earliest_time_db",
        "latest_time_y": "latest_time_db",
        "score": "variable_match_score",

    }
)

# Convert to datetime if they’re strings
for col in ["earliest_time_raw", "latest_time_raw", "earliest_time_db", "latest_time_db"]:
    raw_summary_enriched[col] = pd.to_datetime(raw_summary_enriched[col], errors="coerce")

# Calculate time gaps (in days)
raw_summary_enriched["earliest_diff_days"] = (
    (raw_summary_enriched["earliest_time_raw"] - raw_summary_enriched["earliest_time_db"]).dt.days
)

raw_summary_enriched["latest_diff_days"] = (
    (raw_summary_enriched["latest_time_raw"] - raw_summary_enriched["latest_time_db"]).dt.days
)
raw_summary_enriched = raw_summary_enriched.drop_duplicates()



raw_summary_enriched = raw_summary_enriched.drop(columns=['native_id_db'])

# Save
output_path = os.path.join(output_folder, "validation_Fern_raw_db_station_matched.csv")
raw_summary_enriched.to_csv(output_path, index=False)

print(f"✅ Done. Matches saved to {output_path}")

✅ Done. Matches saved to /workspaces/crmprtd/fern/FERNNorth2025_insert/output/validation_Fern_raw_db_station_matched.csv


In [ ]:
raw_summary_enriched.loc[raw_summary_enriched['latest_diff_days'] > 0]

,station_name,native_id_raw,station_file_name,lat,lon,elev,variable,unit_raw,earliest_time_raw,latest_time_raw,unit_origin,db_var_match,variable_match_score,unit_db,earliest_time_db,latest_time_db,earliest_diff_days,latest_diff_days
